# Norway GIP quad-map backtest

Walk-forward backtest of the Hedgeye-style growth/inflation quad model:
*how often would it have called the correct quad 1-4 quarters ahead, using
only information available at the time?*

**How to run:** `Runtime -> Run all`. That's it.

- The notebook clones the repo, installs dependencies, pulls **real data**
  (SSB mainland GDP + CPI, Norges Bank I-44 and USDNOK, Brent from FRED),
  runs the backtest and renders the results below.
- The repo is private, so the clone step will ask for a GitHub **personal
  access token** the first time (github.com -> Settings -> Developer
  settings -> Fine-grained tokens; read access to this repo is enough).
- If any live fetch fails, the notebook falls back to the synthetic demo
  bundle so you still get a full end-to-end run (clearly labelled).
- GDP revisions: without a Norges Bank real-time vintage file the harness
  runs in the flagged **revision-noise** mode (simulated first releases).
  Upload `data/gdp_vintages.csv` (format in `backtest/fetch_data.py`) to
  switch to true vintages automatically.


In [ ]:
# ---- Parameters: edit and re-run ------------------------------------------
START = "2012"       # first as-of year (or date like "2013-06-30")
END = "2025"         # last as-of year
HORIZON = 4          # predict 1..HORIZON quarters ahead
FREQ = "M"           # as-of dates: "M" month-end (slower, more points) or "Q"
REVISION_SIGMA = 0.25  # noise mode: stdev of first-release QoQ revision (pp)
USE_DEMO = False     # True = skip live data, run on the synthetic bundle


In [ ]:
# ---- Setup: clone the repo and install dependencies ------------------------
import os, subprocess, sys
from pathlib import Path

REPO = "SanderHeisan/Inflation-GDP"
BRANCH = "claude/norway-gip-backtest-harness-nln7uw"

if not Path("backtest.py").exists():          # fresh Colab runtime
    if not Path("Inflation-GDP").exists():
        url = f"https://github.com/{REPO}.git"
        r = subprocess.run(["git", "clone", "-b", BRANCH, url],
                           capture_output=True, text=True)
        if r.returncode != 0:                 # private repo -> need a token
            print("Anonymous clone failed (private repo).")
            from getpass import getpass
            token = getpass("Paste a GitHub personal access token: ").strip()
            url = f"https://{token}@github.com/{REPO}.git"
            subprocess.run(["git", "clone", "-b", BRANCH, url], check=True)
    os.chdir("Inflation-GDP")

print("working dir:", Path.cwd())
%pip install -q -r requirements.txt
print("setup done")


In [ ]:
# ---- Data: fetch real series (falls back to the synthetic demo) ------------
if not USE_DEMO:
    try:
        from backtest import fetch_data
        fetch_data.main(["--data-dir", "data"])
        print("\nlive data ready in data/")
    except Exception as e:
        print(f"\nLIVE FETCH FAILED ({type(e).__name__}: {e})")
        print("Falling back to the synthetic demo bundle. Results below are")
        print("a harness validation, NOT Norway. Common causes: SSB changed")
        print("a table/variable code (see quadmap/data_sources.py notes) or")
        print("a temporary API outage - just re-run this cell to retry.")
        USE_DEMO = True
else:
    print("USE_DEMO=True - skipping live data")


In [ ]:
# ---- Run the walk-forward backtest -----------------------------------------
import subprocess, sys

cmd = [sys.executable, "backtest.py", "--start", str(START), "--end", str(END),
       "--horizon", str(HORIZON), "--freq", FREQ,
       "--sigma", str(REVISION_SIGMA), "--outdir", "results"]
if USE_DEMO:
    cmd.append("--demo")

print(" ".join(cmd), "\n")
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("backtest failed - see stderr above")


In [ ]:
# ---- Headline table: quad hit rate by horizon vs benchmarks ----------------
import pandas as pd

summary = pd.read_csv("results/summary.csv")
hit = (summary.pivot_table(index="horizon", columns=["basis", "strategy"],
                           values="hit_rate")
       .round(3))
print("Quad hit rate (share of quarters called correctly):")
display(hit)

model = summary[summary["strategy"] == "model"].set_index(["basis", "horizon"])
cols = ["n", "hit_rate", "hit_rate_high_conviction", "growth_dir_hit",
        "inflation_dir_hit", "flip_precision", "flip_precision_exact",
        "flip_recall", "edge_vs_persistence", "edge_vs_base_effects",
        "edge_vs_random"]
print("\nModel detail (both scoring bases):")
display(model[cols].round(3))


In [ ]:
# ---- Charts -----------------------------------------------------------------
from pathlib import Path
from IPython.display import Image, display

for png in ["hit_rate_final.png", "hit_rate_first_release.png",
            *sorted(Path("results").glob("confusion_h*.png")),
            "timeline.png"]:
    p = Path("results") / png if isinstance(png, str) else png
    if p.exists():
        display(Image(filename=str(p)))


In [ ]:
# ---- Optional: download everything (parquet + csv + plots) as a zip --------
import shutil

shutil.make_archive("backtest_output", "zip", "results")
try:
    from google.colab import files
    files.download("backtest_output.zip")
except ImportError:
    print("not running in Colab - backtest_output.zip is in the working dir")


## Going further

- **True GDP vintages:** download the Norges Bank real-time database
  (norges-bank.no -> Statistics), convert it to the CSV format documented in
  `backtest/fetch_data.py`, upload it as `data/gdp_vintages.csv` (Colab file
  pane -> drag & drop), and re-run from the backtest cell. The harness
  switches to `revision_mode='realtime'` automatically.
- **Better market history:** `data/market_monthly.csv` ships with proxies
  (the power column is rescaled from the CPI electricity sub-index).
  Replace it with real Nord Pool / Brent / USDNOK monthly averages and
  re-run - same file, same columns.
- **Every individual prediction** is in `results/backtest_results.parquet`
  (as-of date, target quarter, horizon, quad, deltas, conviction flag,
  benchmark calls, revision-mode flag).
